# 09 Threshold Classification Dataset

## Purpose

This notebook constructs the first threshold-classification dataset for Hong Kong temperature markets.

The purpose is to move from the prototype Gaussian probability conversion towards a supervised-learning target that matches the market payoff directly. For a temperature threshold \(K\), the target is whether the official settlement-source daily maximum temperature is at least \(K\).

```math
Y_t^{(K)} = \mathbf{1}\{T_t^{\mathrm{settlement}} \geq K\}
```

Each row in the final dataset represents one forecast information time and one temperature threshold. The official realised temperature is used only to construct the realised target after the event. Market-implied probabilities are sampled using the look-ahead-safe timing rule: the first available Polymarket price at or after the forecast issue time.

This is a prototype dataset for the supervised probability-modelling stage. AI weather forecast features are not yet attached in this notebook and are therefore recorded as pending fields.


## 1. Imports and paths


In [3]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
from io import StringIO
import json
import ast
import re

import requests
import pandas as pd
import numpy as np

RAW_DIR = Path("../data/raw/threshold_dataset")
PROCESSED_DIR = Path("../data/processed/threshold_dataset")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 120)

print("Notebook run time UTC:", datetime.now(timezone.utc).isoformat())
print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)


Notebook run time UTC: 2026-06-17T00:42:22.112088+00:00
Raw directory: ../data/raw/threshold_dataset
Processed directory: ../data/processed/threshold_dataset


## 2. Configuration


In [5]:
contract_metadata = {
    "city": "Hong Kong",
    "local_timezone": "Asia/Hong_Kong",
    "latitude": 22.3027,
    "longitude": 114.1740,
    "contract_slug": "highest-temperature-in-hong-kong-on-may-30-2026",
    "polymarket_event_url": "https://polymarket.com/event/highest-temperature-in-hong-kong-on-may-30-2026",
    "settlement_date_local": "2026-05-30",
    "settlement_source": "Hong Kong Observatory",
    "settlement_variable": "Daily maximum temperature",
    "unit": "deg C",
}

# Example forecast issue times. These will later be replaced by actual AI/weather-model issue times.
forecast_run_config = [
    {
        "forecast_model": "AIFS_or_ECMWF_example",
        "forecast_run_time_utc": "2026-05-28 12:00:00",
        "availability_lag_minutes": 0,
        "forecast_valid_time_utc": "2026-05-30 12:00:00",
        "timing_label": "two_day_valid_time_example",
    },
    {
        "forecast_model": "AIFS_or_ECMWF_example",
        "forecast_run_time_utc": "2026-05-29 00:00:00",
        "availability_lag_minutes": 0,
        "forecast_valid_time_utc": "2026-05-30 12:00:00",
        "timing_label": "one_and_half_day_valid_time_example",
    },
    {
        "forecast_model": "AIFS_or_ECMWF_example",
        "forecast_run_time_utc": "2026-05-29 12:00:00",
        "availability_lag_minutes": 0,
        "forecast_valid_time_utc": "2026-05-30 12:00:00",
        "timing_label": "one_day_valid_time_example",
    },
    {
        "forecast_model": "AIFS_or_ECMWF_example",
        "forecast_run_time_utc": "2026-05-30 00:00:00",
        "availability_lag_minutes": 0,
        "forecast_valid_time_utc": "2026-05-30 12:00:00",
        "timing_label": "same_day_valid_time_example",
    },
]

contract_metadata


{'city': 'Hong Kong',
 'local_timezone': 'Asia/Hong_Kong',
 'latitude': 22.3027,
 'longitude': 114.174,
 'contract_slug': 'highest-temperature-in-hong-kong-on-may-30-2026',
 'polymarket_event_url': 'https://polymarket.com/event/highest-temperature-in-hong-kong-on-may-30-2026',
 'settlement_date_local': '2026-05-30',
 'settlement_source': 'Hong Kong Observatory',
 'settlement_variable': 'Daily maximum temperature',
 'unit': 'deg C'}

## 3. Helper functions


In [7]:
def fetch_text(url, timeout=30, params=None):
    try:
        response = requests.get(url, params=params, timeout=timeout)
        return {
            "url": response.url,
            "status_code": response.status_code,
            "ok": response.ok,
            "content_type": response.headers.get("content-type"),
            "text": response.text,
            "error": None,
        }
    except Exception as exc:
        return {
            "url": url,
            "status_code": None,
            "ok": False,
            "content_type": None,
            "text": "",
            "error": repr(exc),
        }


def safe_json_loads(value):
    if isinstance(value, (list, dict)):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None

    text = str(value)
    try:
        return json.loads(text)
    except Exception:
        try:
            return ast.literal_eval(text)
        except Exception:
            return None


def normalise_col(col):
    return re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")


def parse_utc(dt_str):
    return pd.Timestamp(dt_str, tz="UTC").to_pydatetime()


def local_day_window_utc(local_date_str, timezone_name):
    local_tz = ZoneInfo(timezone_name)
    local_start = datetime.fromisoformat(local_date_str).replace(tzinfo=local_tz)
    local_end = local_start + timedelta(days=1) - timedelta(seconds=1)
    local_midpoint = local_start + timedelta(hours=12)

    return {
        "target_local_date": local_date_str,
        "target_local_day_start": local_start,
        "target_local_day_midpoint": local_midpoint,
        "target_local_day_end": local_end,
        "target_local_day_start_utc": local_start.astimezone(timezone.utc),
        "target_local_day_midpoint_utc": local_midpoint.astimezone(timezone.utc),
        "target_local_day_end_utc": local_end.astimezone(timezone.utc),
    }


def select_first_price_at_or_after(price_history, timestamp_col, price_col, issue_time):
    if price_history.empty:
        return {
            "market_price_time_utc": pd.NaT,
            "market_yes_price": np.nan,
            "price_selection_status": "empty_price_history",
            "price_time_gap_minutes": np.nan,
        }

    df = price_history.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], utc=True, errors="coerce")
    df = df.dropna(subset=[timestamp_col, price_col]).sort_values(timestamp_col)

    issue_ts = pd.Timestamp(issue_time)
    if issue_ts.tzinfo is None:
        issue_ts = issue_ts.tz_localize("UTC")
    else:
        issue_ts = issue_ts.tz_convert("UTC")

    candidates = df[df[timestamp_col] >= issue_ts].sort_values(timestamp_col)

    if candidates.empty:
        return {
            "market_price_time_utc": pd.NaT,
            "market_yes_price": np.nan,
            "price_selection_status": "no_price_after_issue_time",
            "price_time_gap_minutes": np.nan,
        }

    row = candidates.iloc[0]
    gap_minutes = (row[timestamp_col] - issue_ts).total_seconds() / 60

    return {
        "market_price_time_utc": row[timestamp_col],
        "market_yes_price": row[price_col],
        "price_selection_status": "first_price_at_or_after_issue_time",
        "price_time_gap_minutes": gap_minutes,
    }


## 4. Load or retrieve official HKO settlement value


In [9]:
def standardise_hko_daily_max(df, source_name):
    out = df.copy()
    out.columns = [normalise_col(c) for c in out.columns]

    if {"year", "month", "day"}.issubset(set(out.columns)):
        date_series = pd.to_datetime(
            {
                "year": pd.to_numeric(out["year"], errors="coerce"),
                "month": pd.to_numeric(out["month"], errors="coerce"),
                "day": pd.to_numeric(out["day"], errors="coerce"),
            },
            errors="coerce",
        )
    else:
        date_candidates = [c for c in out.columns if c in ["date", "data", "yyyymmdd"] or "date" in c]
        date_col = date_candidates[0] if date_candidates else out.columns[0]
        date_series = pd.to_datetime(out[date_col], errors="coerce")

    temp_candidates = [
        c for c in out.columns
        if ("max" in c and ("temp" in c or "temperature" in c))
        or ("maximum" in c and ("temp" in c or "temperature" in c))
        or c in ["value", "temperature", "temp"]
    ]

    if temp_candidates:
        temp_col = temp_candidates[0]
    else:
        numeric_scores = {}
        for c in out.columns:
            if c in ["year", "month", "day"]:
                continue
            numeric_scores[c] = pd.to_numeric(out[c], errors="coerce").notna().sum()
        temp_col = max(numeric_scores, key=numeric_scores.get)

    standardised = pd.DataFrame({
        "settlement_date_local": date_series.dt.date.astype(str),
        "official_realised_temperature_c": pd.to_numeric(out[temp_col], errors="coerce"),
        "source_name": source_name,
        "source_route": "HKO open data CSV",
    })

    return standardised.dropna(subset=["settlement_date_local", "official_realised_temperature_c"])


def load_or_retrieve_hko_settlement(target_date):
    saved_path = Path("../data/processed/hko/hko_official_settlement_temperature.csv")

    if saved_path.exists():
        saved = pd.read_csv(saved_path)
        if "settlement_date_local" in saved.columns and "official_realised_temperature_c" in saved.columns:
            match = saved[
                (saved["settlement_date_local"].astype(str) == target_date)
                & saved["official_realised_temperature_c"].notna()
            ].copy()

            if not match.empty:
                row = match.iloc[0]
                return pd.DataFrame([{
                    "settlement_date_local": target_date,
                    "official_realised_temperature_c": float(row["official_realised_temperature_c"]),
                    "settlement_source": row.get("settlement_source", "Hong Kong Observatory"),
                    "settlement_variable": row.get("settlement_variable", "Daily maximum temperature"),
                    "source_route": row.get("source_route", "saved HKO settlement table"),
                    "source_reference": row.get("source_reference", str(saved_path)),
                    "retrieval_status": row.get("retrieval_status", "loaded_saved_settlement_table"),
                }])

    year = pd.Timestamp(target_date).year
    urls = {
        "all_year_daily_max": "https://data.weather.gov.hk/weatherAPI/opendata/opendata.php?dataType=CLMMAXT&rformat=csv&station=HKO",
        "target_year_daily_max": f"https://data.weather.gov.hk/weatherAPI/opendata/opendata.php?dataType=CLMMAXT&year={year}&rformat=csv&station=HKO",
    }

    matches = []

    for source_name, url in urls.items():
        result = fetch_text(url)
        print(source_name, result["status_code"], result["content_type"], "chars:", len(result["text"]))

        if not result["ok"] or not result["text"].strip():
            continue

        (RAW_DIR / f"hko_{source_name}.csv").write_text(result["text"], encoding="utf-8")

        try:
            raw_df = pd.read_csv(StringIO(result["text"]))
            standardised = standardise_hko_daily_max(raw_df, source_name)
            match = standardised[standardised["settlement_date_local"] == target_date].copy()
            if not match.empty:
                match["source_reference"] = url
                matches.append(match)
        except Exception as exc:
            print(source_name, "parse failed:", repr(exc))

    if not matches:
        return pd.DataFrame([{
            "settlement_date_local": target_date,
            "official_realised_temperature_c": np.nan,
            "settlement_source": "Hong Kong Observatory",
            "settlement_variable": "Daily maximum temperature",
            "source_route": "not_matched",
            "source_reference": "",
            "retrieval_status": "not_found",
        }])

    combined = pd.concat(matches, ignore_index=True)
    combined = combined.drop_duplicates(subset=["settlement_date_local", "official_realised_temperature_c"])
    row = combined.iloc[0]

    return pd.DataFrame([{
        "settlement_date_local": target_date,
        "official_realised_temperature_c": float(row["official_realised_temperature_c"]),
        "settlement_source": "Hong Kong Observatory",
        "settlement_variable": "Daily maximum temperature",
        "source_route": row["source_route"],
        "source_reference": row["source_reference"],
        "retrieval_status": "matched_open_data_csv",
    }])


settlement_df = load_or_retrieve_hko_settlement(contract_metadata["settlement_date_local"])
display(settlement_df)

official_realised_temperature_c = settlement_df["official_realised_temperature_c"].iloc[0]
print("Official realised temperature:", official_realised_temperature_c)
print("Retrieval status:", settlement_df["retrieval_status"].iloc[0])


,settlement_date_local,official_realised_temperature_c,settlement_source,settlement_variable,source_route,source_reference,retrieval_status
0,2026-05-30,32.6,Hong Kong Observatory,Daily Maximum Temperature,HKO open data CSV,https://data.weather.gov.hk/weatherAPI/opendat...,matched_open_data_csv


Official realised temperature: 32.6
Retrieval status: matched_open_data_csv


## 5. Build forecast timing table


In [11]:
window = local_day_window_utc(
    contract_metadata["settlement_date_local"],
    contract_metadata["local_timezone"],
)

forecast_timing_df = pd.DataFrame(forecast_run_config)

forecast_timing_df["forecast_run_time_utc"] = forecast_timing_df["forecast_run_time_utc"].apply(parse_utc)
forecast_timing_df["forecast_valid_time_utc"] = forecast_timing_df["forecast_valid_time_utc"].apply(parse_utc)

forecast_timing_df["forecast_issue_time_utc"] = (
    forecast_timing_df["forecast_run_time_utc"]
    + pd.to_timedelta(forecast_timing_df["availability_lag_minutes"], unit="m")
)

forecast_timing_df["target_local_date"] = contract_metadata["settlement_date_local"]
forecast_timing_df["target_local_day_start_utc"] = window["target_local_day_start_utc"]
forecast_timing_df["target_local_day_midpoint_utc"] = window["target_local_day_midpoint_utc"]
forecast_timing_df["target_local_day_end_utc"] = window["target_local_day_end_utc"]

forecast_timing_df["lead_time_to_valid_hours"] = (
    forecast_timing_df["forecast_valid_time_utc"] - forecast_timing_df["forecast_issue_time_utc"]
).dt.total_seconds() / 3600

forecast_timing_df["lead_time_to_day_start_hours"] = (
    forecast_timing_df["target_local_day_start_utc"] - forecast_timing_df["forecast_issue_time_utc"]
).dt.total_seconds() / 3600

forecast_timing_df["lead_time_to_day_midpoint_hours"] = (
    forecast_timing_df["target_local_day_midpoint_utc"] - forecast_timing_df["forecast_issue_time_utc"]
).dt.total_seconds() / 3600

forecast_timing_df["lead_time_to_day_end_hours"] = (
    forecast_timing_df["target_local_day_end_utc"] - forecast_timing_df["forecast_issue_time_utc"]
).dt.total_seconds() / 3600

forecast_timing_df["forecast_issue_before_local_day_start"] = (
    forecast_timing_df["forecast_issue_time_utc"] <= forecast_timing_df["target_local_day_start_utc"]
)

forecast_timing_df["forecast_issue_before_local_day_midpoint"] = (
    forecast_timing_df["forecast_issue_time_utc"] <= forecast_timing_df["target_local_day_midpoint_utc"]
)

forecast_timing_df["forecast_issue_before_local_day_end"] = (
    forecast_timing_df["forecast_issue_time_utc"] <= forecast_timing_df["target_local_day_end_utc"]
)

forecast_timing_df["forecast_timing_category"] = np.select(
    [
        forecast_timing_df["forecast_issue_before_local_day_start"],
        forecast_timing_df["forecast_issue_before_local_day_end"],
    ],
    [
        "full_day_ex_ante",
        "within_day_update",
    ],
    default="after_target_day",
)

forecast_timing_df["market_price_timestamp_rule"] = (
    "first available Polymarket price at or after forecast_issue_time_utc"
)

display(forecast_timing_df)


,forecast_model,forecast_run_time_utc,availability_lag_minutes,forecast_valid_time_utc,timing_label,forecast_issue_time_utc,target_local_date,target_local_day_start_utc,target_local_day_midpoint_utc,target_local_day_end_utc,lead_time_to_valid_hours,lead_time_to_day_start_hours,lead_time_to_day_midpoint_hours,lead_time_to_day_end_hours,forecast_issue_before_local_day_start,forecast_issue_before_local_day_midpoint,forecast_issue_before_local_day_end,forecast_timing_category,market_price_timestamp_rule
0,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,0,2026-05-30 12:00:00+00:00,two_day_valid_time_example,2026-05-28 12:00:00+00:00,2026-05-30,2026-05-29 16:00:00+00:00,2026-05-30 04:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,True,True,True,full_day_ex_ante,first available Polymarket price at or after f...
1,AIFS_or_ECMWF_example,2026-05-29 00:00:00+00:00,0,2026-05-30 12:00:00+00:00,one_and_half_day_valid_time_example,2026-05-29 00:00:00+00:00,2026-05-30,2026-05-29 16:00:00+00:00,2026-05-30 04:00:00+00:00,2026-05-30 15:59:59+00:00,36.0,16.0,28.0,39.999722,True,True,True,full_day_ex_ante,first available Polymarket price at or after f...
2,AIFS_or_ECMWF_example,2026-05-29 12:00:00+00:00,0,2026-05-30 12:00:00+00:00,one_day_valid_time_example,2026-05-29 12:00:00+00:00,2026-05-30,2026-05-29 16:00:00+00:00,2026-05-30 04:00:00+00:00,2026-05-30 15:59:59+00:00,24.0,4.0,16.0,27.999722,True,True,True,full_day_ex_ante,first available Polymarket price at or after f...
3,AIFS_or_ECMWF_example,2026-05-30 00:00:00+00:00,0,2026-05-30 12:00:00+00:00,same_day_valid_time_example,2026-05-30 00:00:00+00:00,2026-05-30,2026-05-29 16:00:00+00:00,2026-05-30 04:00:00+00:00,2026-05-30 15:59:59+00:00,12.0,-8.0,4.0,15.999722,False,True,True,within_day_update,first available Polymarket price at or after f...


## 6. Retrieve Polymarket event metadata and parse temperature-bin markets


In [13]:
def fetch_polymarket_event_by_slug(slug):
    urls = [
        f"https://gamma-api.polymarket.com/events/slug/{slug}",
        f"https://gamma-api.polymarket.com/events?slug={slug}",
    ]

    for url in urls:
        try:
            response = requests.get(url, timeout=30)
            print("URL:", url, "status:", response.status_code)

            if not response.ok:
                continue

            data = response.json()

            if isinstance(data, list) and data:
                return data[0], response.url
            if isinstance(data, dict):
                return data, response.url

        except Exception as exc:
            print("Failed:", url, repr(exc))

    return None, None


def extract_yes_markets_from_event(event_data):
    if event_data is None:
        return pd.DataFrame()

    rows = []

    for market in event_data.get("markets", []):
        outcomes = safe_json_loads(market.get("outcomes"))
        outcome_prices = safe_json_loads(market.get("outcomePrices"))
        clob_token_ids = safe_json_loads(market.get("clobTokenIds"))

        if not isinstance(outcomes, list) or not isinstance(clob_token_ids, list):
            continue

        for i, outcome in enumerate(outcomes):
            if str(outcome).upper() != "YES":
                continue

            price = np.nan
            if isinstance(outcome_prices, list) and i < len(outcome_prices):
                price = pd.to_numeric(outcome_prices[i], errors="coerce")

            rows.append({
                "event_slug": contract_metadata["contract_slug"],
                "market_id": market.get("id"),
                "condition_id": market.get("conditionId"),
                "question": market.get("question"),
                "market_slug": market.get("slug"),
                "outcome": outcome,
                "yes_token_id": clob_token_ids[i] if i < len(clob_token_ids) else None,
                "current_yes_price": price,
                "active": market.get("active"),
                "closed": market.get("closed"),
                "end_date": market.get("endDate"),
                "volume": market.get("volume"),
                "liquidity": market.get("liquidity"),
            })

    return pd.DataFrame(rows)


def parse_temperature_bin(question, market_slug=None):
    """
    Parse Polymarket temperature-bin labels.

    The parser deliberately focuses on the temperature phrase and avoids using
    event-date numbers such as "May 30" or "2026". For a single integer label
    such as 32C, the bin is interpreted as [32, 33). For a lower-tail label
    such as 23C or lower, the bin is interpreted as (-inf, 24). For an upper-tail
    label such as 33C or higher, the bin is interpreted as [33, inf).
    """
    question_text = "" if question is None else str(question).lower()
    slug_text = "" if market_slug is None else str(market_slug).lower()
    text = f"{question_text} {slug_text}"

    text = (
        text.replace("°", "")
        .replace("degrees", "")
        .replace("degree", "")
        .replace("celsius", "c")
        .replace("deg", "")
        .replace(",", " ")
    )

    # Prefer slug suffixes because they usually isolate the temperature label.
    # Examples: "...-32c", "...-33c-or-higher", "...-23c-or-lower".
    slug_patterns = [
        (r"-(\d+(?:\.\d+)?)c-(?:or-)?(?:lower|less|below|under)$", "lower_tail"),
        (r"-(\d+(?:\.\d+)?)c-(?:or-)?(?:higher|more|above|over)$", "upper_tail"),
        (r"-(\d+(?:\.\d+)?)c$", "single_degree"),
    ]

    for pattern, label_type in slug_patterns:
        match = re.search(pattern, slug_text)
        if match:
            value = float(match.group(1))

            if label_type == "lower_tail":
                return pd.Series({
                    "bin_lower_c": -np.inf,
                    "bin_upper_c": value + 1,
                    "bin_parse_status": "parsed_lower_tail_slug",
                })

            if label_type == "upper_tail":
                return pd.Series({
                    "bin_lower_c": value,
                    "bin_upper_c": np.inf,
                    "bin_parse_status": "parsed_upper_tail_slug",
                })

            return pd.Series({
                "bin_lower_c": value,
                "bin_upper_c": value + 1,
                "bin_parse_status": "parsed_single_degree_slug",
            })

    # Question-text lower tail: "23C or lower", "23C or less".
    lower_tail_match = re.search(
        r"(\d+(?:\.\d+)?)\s*c?\s*(?:or\s+)?(?:lower|less|below|under)",
        text,
    )
    if lower_tail_match:
        value = float(lower_tail_match.group(1))
        return pd.Series({
            "bin_lower_c": -np.inf,
            "bin_upper_c": value + 1,
            "bin_parse_status": "parsed_lower_tail_question",
        })

    # Question-text upper tail: "33C or higher", "33C or more".
    upper_tail_match = re.search(
        r"(\d+(?:\.\d+)?)\s*c?\s*(?:or\s+)?(?:higher|more|above|over)",
        text,
    )
    if upper_tail_match:
        value = float(upper_tail_match.group(1))
        return pd.Series({
            "bin_lower_c": value,
            "bin_upper_c": np.inf,
            "bin_parse_status": "parsed_upper_tail_question",
        })

    # Exact single-degree bin: "be 32C". This avoids the later date number.
    exact_match = re.search(
        r"be\s+(\d+(?:\.\d+)?)\s*c\b",
        text,
    )
    if exact_match:
        value = float(exact_match.group(1))
        return pd.Series({
            "bin_lower_c": value,
            "bin_upper_c": value + 1,
            "bin_parse_status": "parsed_single_degree_question",
        })

    return pd.Series({
        "bin_lower_c": np.nan,
        "bin_upper_c": np.nan,
        "bin_parse_status": "unparsed",
    })


def format_bin_label(lower, upper):
    if pd.isna(lower) or pd.isna(upper):
        return "unparsed"

    if np.isneginf(lower):
        return f"<{upper:g}"

    if np.isposinf(upper):
        return f">={lower:g}"

    return f"[{lower:g},{upper:g})"


event_data, event_source_url = fetch_polymarket_event_by_slug(contract_metadata["contract_slug"])

if event_data is not None:
    (RAW_DIR / "polymarket_event_metadata.json").write_text(json.dumps(event_data, indent=2), encoding="utf-8")
    print("Event title:", event_data.get("title") or event_data.get("question"))
    print("Number of markets:", len(event_data.get("markets", [])))
else:
    print("No event metadata retrieved.")

yes_markets_df = extract_yes_markets_from_event(event_data)

if not yes_markets_df.empty:
    yes_markets_df["bin_parse_text"] = (
        yes_markets_df["question"].fillna("") + " " + yes_markets_df["market_slug"].fillna("")
    )

    parsed_bins = yes_markets_df.apply(
    lambda row: parse_temperature_bin(row["question"], row["market_slug"]),
    axis=1,)
    yes_markets_df = pd.concat([yes_markets_df.reset_index(drop=True), parsed_bins.reset_index(drop=True)], axis=1)

    yes_markets_df["temperature_bin_label"] = yes_markets_df.apply(
        lambda row: format_bin_label(row["bin_lower_c"], row["bin_upper_c"]), axis=1
    )

    yes_markets_df = (
        yes_markets_df
        .sort_values(["bin_lower_c", "bin_upper_c", "market_id"], na_position="last")
        .reset_index(drop=True)
    )

display(
    yes_markets_df[
        [
            "market_id",
            "question",
            "market_slug",
            "yes_token_id",
            "current_yes_price",
            "bin_lower_c",
            "bin_upper_c",
            "temperature_bin_label",
            "bin_parse_status",
        ]
    ]
)


URL: https://gamma-api.polymarket.com/events/slug/highest-temperature-in-hong-kong-on-may-30-2026 status: 200
Event title: Highest temperature in Hong Kong on May 30?
Number of markets: 11


,market_id,question,market_slug,yes_token_id,current_yes_price,bin_lower_c,bin_upper_c,temperature_bin_label,bin_parse_status
0,2375825,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-may-30-202...,2082456913986935254241388444978012953023591387...,0,-inf,24.0,<24,parsed_lower_tail_question
1,2375826,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-may-30-202...,2943377179678179196313475284772204339317682077...,0,24.0,25.0,"[24,25)",parsed_single_degree_slug
2,2375827,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-may-30-202...,4617896828781800154535996881024729722157020410...,0,25.0,26.0,"[25,26)",parsed_single_degree_slug
3,2375828,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-may-30-202...,1040985512240410515356525715019651610521841919...,0,26.0,27.0,"[26,27)",parsed_single_degree_slug
4,2375829,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-may-30-202...,8612028584163479635813164643298765384451615551...,0,27.0,28.0,"[27,28)",parsed_single_degree_slug
5,2375830,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-may-30-202...,6762641211374974709079728339036642145194676044...,0,28.0,29.0,"[28,29)",parsed_single_degree_slug
6,2375831,Will the highest temperature in Hong Kong be 2...,highest-temperature-in-hong-kong-on-may-30-202...,1264519449661180374335059763317437131424227980...,0,29.0,30.0,"[29,30)",parsed_single_degree_slug
7,2375832,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-may-30-202...,1030906326133351056965750446522265743053310917...,0,30.0,31.0,"[30,31)",parsed_single_degree_slug
8,2375833,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-may-30-202...,1866150743015618681650630480883665441327029368...,0,31.0,32.0,"[31,32)",parsed_single_degree_slug
9,2375834,Will the highest temperature in Hong Kong be 3...,highest-temperature-in-hong-kong-on-may-30-202...,3292528861173607221297409026900055602250951713...,1,32.0,33.0,"[32,33)",parsed_single_degree_slug


## 7. Retrieve Polymarket YES price histories


In [15]:
def fetch_price_history(token_id, interval="max"):
    if token_id is None or str(token_id) == "nan":
        return pd.DataFrame()

    url = "https://clob.polymarket.com/prices-history"
    params = {"market": str(token_id), "interval": interval}

    try:
        response = requests.get(url, params=params, timeout=30)

        if not response.ok:
            return pd.DataFrame({
                "yes_token_id": [str(token_id)],
                "history_status": [f"http_{response.status_code}"],
                "history_error": [response.text[:300]],
            })

        data = response.json()
        history = data.get("history", data if isinstance(data, list) else [])

        rows = []
        for item in history:
            if not isinstance(item, dict):
                continue
            rows.append({
                "yes_token_id": str(token_id),
                "timestamp_raw": item["t"] if "t" in item else item.get("timestamp"),
                "price": item["p"] if "p" in item else item.get("price"),
                "history_status": "ok",
                "history_error": None,
            })

        hist = pd.DataFrame(rows)

        if hist.empty:
            return hist

        hist["timestamp_utc"] = pd.to_datetime(hist["timestamp_raw"], unit="s", utc=True, errors="coerce")
        hist["price"] = pd.to_numeric(hist["price"], errors="coerce")
        hist = hist.dropna(subset=["timestamp_utc", "price"]).sort_values("timestamp_utc")

        return hist

    except Exception as exc:
        return pd.DataFrame({
            "yes_token_id": [str(token_id)],
            "history_status": ["exception"],
            "history_error": [repr(exc)],
        })


price_history_frames = []

for token_id in yes_markets_df["yes_token_id"].dropna().unique() if not yes_markets_df.empty else []:
    hist = fetch_price_history(token_id)
    if not hist.empty:
        price_history_frames.append(hist)

price_history_df = pd.concat(price_history_frames, ignore_index=True) if price_history_frames else pd.DataFrame()

print("Price history rows:", len(price_history_df))
display(price_history_df.head())
display(price_history_df.tail())


Price history rows: 6727


,yes_token_id,timestamp_raw,price,history_status,history_error,timestamp_utc
0,2082456913986935254241388444978012953023591387...,1779942004,0.0050,ok,None,2026-05-28 04:20:04+00:00
1,2082456913986935254241388444978012953023591387...,1779942605,0.0050,ok,None,2026-05-28 04:30:05+00:00
2,2082456913986935254241388444978012953023591387...,1779943205,0.0025,ok,None,2026-05-28 04:40:05+00:00
3,2082456913986935254241388444978012953023591387...,1779943807,0.0025,ok,None,2026-05-28 04:50:07+00:00
4,2082456913986935254241388444978012953023591387...,1779944408,0.0025,ok,None,2026-05-28 05:00:08+00:00


,yes_token_id,timestamp_raw,price,history_status,history_error,timestamp_utc
6722,7673945377828659498594688521559031179409557337...,1780304405,0.0005,ok,None,2026-06-01 09:00:05+00:00
6723,7673945377828659498594688521559031179409557337...,1780305004,0.0005,ok,None,2026-06-01 09:10:04+00:00
6724,7673945377828659498594688521559031179409557337...,1780305605,0.0005,ok,None,2026-06-01 09:20:05+00:00
6725,7673945377828659498594688521559031179409557337...,1780306204,0.0005,ok,None,2026-06-01 09:30:04+00:00
6726,7673945377828659498594688521559031179409557337...,1780306805,0.0005,ok,None,2026-06-01 09:40:05+00:00


## 8. Select bin prices at each forecast issue time


In [17]:
bin_price_rows = []

for _, timing_row in forecast_timing_df.iterrows():
    issue_time = timing_row["forecast_issue_time_utc"]

    for _, market_row in yes_markets_df.iterrows():
        token_id = str(market_row["yes_token_id"])

        if not price_history_df.empty and "yes_token_id" in price_history_df.columns:
            token_hist = price_history_df[price_history_df["yes_token_id"].astype(str) == token_id].copy()
        else:
            token_hist = pd.DataFrame()

        selected = select_first_price_at_or_after(
            token_hist,
            timestamp_col="timestamp_utc",
            price_col="price",
            issue_time=issue_time,
        )

        bin_price_rows.append({
            "contract_slug": contract_metadata["contract_slug"],
            "city": contract_metadata["city"],
            "target_local_date": contract_metadata["settlement_date_local"],
            "forecast_model": timing_row["forecast_model"],
            "timing_label": timing_row["timing_label"],
            "forecast_timing_category": timing_row["forecast_timing_category"],
            "forecast_run_time_utc": timing_row["forecast_run_time_utc"],
            "forecast_issue_time_utc": timing_row["forecast_issue_time_utc"],
            "forecast_valid_time_utc": timing_row["forecast_valid_time_utc"],
            "target_local_day_start_utc": timing_row["target_local_day_start_utc"],
            "target_local_day_midpoint_utc": timing_row["target_local_day_midpoint_utc"],
            "target_local_day_end_utc": timing_row["target_local_day_end_utc"],
            "lead_time_to_valid_hours": timing_row["lead_time_to_valid_hours"],
            "lead_time_to_day_start_hours": timing_row["lead_time_to_day_start_hours"],
            "lead_time_to_day_midpoint_hours": timing_row["lead_time_to_day_midpoint_hours"],
            "lead_time_to_day_end_hours": timing_row["lead_time_to_day_end_hours"],
            "market_id": market_row["market_id"],
            "market_slug": market_row["market_slug"],
            "question": market_row["question"],
            "yes_token_id": market_row["yes_token_id"],
            "temperature_bin_label": market_row["temperature_bin_label"],
            "bin_lower_c": market_row["bin_lower_c"],
            "bin_upper_c": market_row["bin_upper_c"],
            "market_price_time_utc": selected["market_price_time_utc"],
            "market_yes_price": selected["market_yes_price"],
            "price_selection_status": selected["price_selection_status"],
            "price_time_gap_minutes": selected["price_time_gap_minutes"],
            "market_price_timestamp_rule": timing_row["market_price_timestamp_rule"],
        })

bin_price_df = pd.DataFrame(bin_price_rows)

if not bin_price_df.empty:
    bin_price_df["forecast_issue_time_utc"] = pd.to_datetime(bin_price_df["forecast_issue_time_utc"], utc=True)
    bin_price_df["market_price_time_utc"] = pd.to_datetime(bin_price_df["market_price_time_utc"], utc=True, errors="coerce")
    bin_price_df["price_after_or_at_issue"] = bin_price_df["market_price_time_utc"] >= bin_price_df["forecast_issue_time_utc"]

    bin_price_df["market_bin_probability_sum_raw"] = (
        bin_price_df.groupby("timing_label")["market_yes_price"].transform("sum")
    )

    bin_price_df["market_bin_probability_normalised"] = np.where(
        bin_price_df["market_bin_probability_sum_raw"] > 0,
        bin_price_df["market_yes_price"] / bin_price_df["market_bin_probability_sum_raw"],
        np.nan,
    )

display(bin_price_df.head(30))

if not bin_price_df.empty:
    print("All selected prices at or after issue time:", bin_price_df["price_after_or_at_issue"].dropna().all())
    display(
        bin_price_df.groupby("timing_label")
        .agg(
            selected_bins=("market_yes_price", "count"),
            raw_probability_sum=("market_yes_price", "sum"),
            average_price_gap_minutes=("price_time_gap_minutes", "mean"),
        )
        .reset_index()
    )


,contract_slug,city,target_local_date,forecast_model,timing_label,forecast_timing_category,forecast_run_time_utc,forecast_issue_time_utc,forecast_valid_time_utc,target_local_day_start_utc,target_local_day_midpoint_utc,target_local_day_end_utc,lead_time_to_valid_hours,lead_time_to_day_start_hours,lead_time_to_day_midpoint_hours,lead_time_to_day_end_hours,market_id,market_slug,question,yes_token_id,temperature_bin_label,bin_lower_c,bin_upper_c,market_price_time_utc,market_yes_price,price_selection_status,price_time_gap_minutes,market_price_timestamp_rule,price_after_or_at_issue,market_bin_probability_sum_raw,market_bin_probability_normalised
0,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,2026-05-29 16:00:00+00:00,2026-05-30 04:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,2375825,highest-temperature-in-hong-kong-on-may-30-202...,Will the highest temperature in Hong Kong be 2...,2082456913986935254241388444978012953023591387...,<24,-inf,24.0,2026-05-28 12:00:07+00:00,0.0010,first_price_at_or_after_issue_time,0.116667,first available Polymarket price at or after f...,True,0.9655,0.001036
1,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,2026-05-29 16:00:00+00:00,2026-05-30 04:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,2375826,highest-temperature-in-hong-kong-on-may-30-202...,Will the highest temperature in Hong Kong be 2...,2943377179678179196313475284772204339317682077...,"[24,25)",24.0,25.0,2026-05-28 12:00:12+00:00,0.0025,first_price_at_or_after_issue_time,0.200000,first available Polymarket price at or after f...,True,0.9655,0.002589
2,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,2026-05-29 16:00:00+00:00,2026-05-30 04:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,2375827,highest-temperature-in-hong-kong-on-may-30-202...,Will the highest temperature in Hong Kong be 2...,4617896828781800154535996881024729722157020410...,"[25,26)",25.0,26.0,2026-05-28 12:00:12+00:00,0.0025,first_price_at_or_after_issue_time,0.200000,first available Polymarket price at or after f...,True,0.9655,0.002589
3,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,2026-05-29 16:00:00+00:00,2026-05-30 04:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,2375828,highest-temperature-in-hong-kong-on-may-30-202...,Will the highest temperature in Hong Kong be 2...,1040985512240410515356525715019651610521841919...,"[26,27)",26.0,27.0,2026-05-28 12:00:04+00:00,0.0055,first_price_at_or_after_issue_time,0.066667,first available Polymarket price at or after f...,True,0.9655,0.005697
4,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,2026-05-29 16:00:00+00:00,2026-05-30 04:00:00+00:00,2026-05-30 15:59:59+00:00,48.0,28.0,40.0,51.999722,2375829,highest-temperature-in-hong-kong-on-may-30-202...,Will the highest temperature in Hong Kong be 2...,8612028584163479635813164643298765384451615551...,"[27,28)",27.0,28.0,2026-05-28 12:00:04+00:00,0.0035,first_price_at_or_after_issue_time,0.066667,first available Polymarket price at or after f...,True,0.9655,0.003625
5,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,AIFS_or_ECMWF_example,two_day_valid_time_example,full_day_ex_ante,2

All selected prices at or after issue time: True


,timing_label,selected_bins,raw_probability_sum,average_price_gap_minutes
0,one_and_half_day_valid_time_example,11,1.0450,0.150000
1,one_day_valid_time_example,11,1.0570,10.053030
2,same_day_valid_time_example,11,1.0410,0.321212
3,two_day_valid_time_example,11,0.9655,0.100000


## 9. Construct threshold-classification dataset


In [19]:
def infer_threshold_grid(bin_price_df):
    if bin_price_df.empty:
        return []

    finite_lowers = sorted(
        set(
            float(value)
            for value in bin_price_df["bin_lower_c"].dropna().unique()
            if np.isfinite(value)
        )
    )

    return finite_lowers


def bin_contributes_to_threshold(lower, upper, threshold):
    if pd.isna(lower) or pd.isna(upper):
        return False

    # For thresholds aligned with bin boundaries, the bins with lower bound at or above
    # the threshold are fully included in the exceedance event.
    return lower >= threshold


threshold_grid = infer_threshold_grid(bin_price_df)
print("Threshold grid:", threshold_grid)

threshold_rows = []

for _, timing_row in forecast_timing_df.iterrows():
    timing_label = timing_row["timing_label"]
    timing_bins = bin_price_df[bin_price_df["timing_label"] == timing_label].copy()

    for threshold_c in threshold_grid:
        contributing_bins = timing_bins[
            timing_bins.apply(
                lambda row: bin_contributes_to_threshold(row["bin_lower_c"], row["bin_upper_c"], threshold_c),
                axis=1,
            )
        ].copy()

        realised_exceeds_threshold = (
            int(official_realised_temperature_c >= threshold_c)
            if pd.notna(official_realised_temperature_c)
            else np.nan
        )

        threshold_rows.append({
            "contract_slug": contract_metadata["contract_slug"],
            "city": contract_metadata["city"],
            "target_local_date": contract_metadata["settlement_date_local"],
            "settlement_source": settlement_df["settlement_source"].iloc[0],
            "settlement_variable": settlement_df["settlement_variable"].iloc[0],
            "official_realised_temperature_c": official_realised_temperature_c,
            "threshold_c": threshold_c,
            "realised_exceeds_threshold": realised_exceeds_threshold,
            "forecast_model": timing_row["forecast_model"],
            "forecast_run_time_utc": timing_row["forecast_run_time_utc"],
            "forecast_issue_time_utc": timing_row["forecast_issue_time_utc"],
            "forecast_valid_time_utc": timing_row["forecast_valid_time_utc"],
            "timing_label": timing_label,
            "forecast_timing_category": timing_row["forecast_timing_category"],
            "lead_time_to_valid_hours": timing_row["lead_time_to_valid_hours"],
            "lead_time_to_day_start_hours": timing_row["lead_time_to_day_start_hours"],
            "lead_time_to_day_midpoint_hours": timing_row["lead_time_to_day_midpoint_hours"],
            "lead_time_to_day_end_hours": timing_row["lead_time_to_day_end_hours"],
            "market_implied_threshold_prob_raw": contributing_bins["market_yes_price"].sum(),
            "market_implied_threshold_prob_normalised": contributing_bins["market_bin_probability_normalised"].sum(),
            "number_of_contributing_bins": len(contributing_bins),
            "average_price_time_gap_minutes": timing_bins["price_time_gap_minutes"].mean(),
            "market_price_timestamp_rule": timing_row["market_price_timestamp_rule"],
            "forecast_temperature_c": np.nan,
            "forecast_temperature_source": "not_attached_in_step_7",
            "forecast_feature_status": "pending_ai_weather_feature_merge",
        })

threshold_dataset_df = pd.DataFrame(threshold_rows)

display(threshold_dataset_df)


Threshold grid: [24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0]


,contract_slug,city,target_local_date,settlement_source,settlement_variable,official_realised_temperature_c,threshold_c,realised_exceeds_threshold,forecast_model,forecast_run_time_utc,forecast_issue_time_utc,forecast_valid_time_utc,timing_label,forecast_timing_category,lead_time_to_valid_hours,lead_time_to_day_start_hours,lead_time_to_day_midpoint_hours,lead_time_to_day_end_hours,market_implied_threshold_prob_raw,market_implied_threshold_prob_normalised,number_of_contributing_bins,average_price_time_gap_minutes,market_price_timestamp_rule,forecast_temperature_c,forecast_temperature_source,forecast_feature_status
0,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,24.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9645,0.998964,10,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge
1,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,25.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9620,0.996375,9,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge
2,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,26.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9595,0.993786,8,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge
3,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,27.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9540,0.988089,7,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge
4,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,28.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9505,0.984464,6,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge
5,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,29.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9155,0.948213,5,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge
6,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,30.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.8700,0.901088,4,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge
7,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,31.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.7050,0.730192,3,0.

## 10. Diagnostics


In [21]:
print("Bin price table shape:", bin_price_df.shape)
print("Threshold dataset shape:", threshold_dataset_df.shape)

if not yes_markets_df.empty:
    print("Parsed bin statuses:")
    display(yes_markets_df["bin_parse_status"].value_counts(dropna=False))

if not threshold_dataset_df.empty:
    print("Unique timing labels:", threshold_dataset_df["timing_label"].nunique())
    print("Unique thresholds:", threshold_dataset_df["threshold_c"].nunique())

    print("Target value counts:")
    display(threshold_dataset_df["realised_exceeds_threshold"].value_counts(dropna=False))

    print("Market-implied threshold probability range:")
    display(
        threshold_dataset_df[
            [
                "market_implied_threshold_prob_raw",
                "market_implied_threshold_prob_normalised",
                "average_price_time_gap_minutes",
            ]
        ].describe()
    )

    display(
        threshold_dataset_df[
            [
                "timing_label",
                "forecast_timing_category",
                "threshold_c",
                "official_realised_temperature_c",
                "realised_exceeds_threshold",
                "market_implied_threshold_prob_normalised",
                "lead_time_to_valid_hours",
                "average_price_time_gap_minutes",
                "forecast_feature_status",
            ]
        ].head(40)
    )

# Basic validation flags.
validation_summary = {
    "has_official_realised_temperature": pd.notna(official_realised_temperature_c),
    "has_polymarket_event": event_data is not None,
    "has_yes_markets": not yes_markets_df.empty,
    "has_price_history": not price_history_df.empty,
    "has_bin_price_table": not bin_price_df.empty,
    "has_threshold_dataset": not threshold_dataset_df.empty,
}

validation_summary


Bin price table shape: (44, 31)
Threshold dataset shape: (40, 26)
Parsed bin statuses:


bin_parse_status
parsed_single_degree_slug     9
parsed_lower_tail_question    1
parsed_upper_tail_question    1
Name: count, dtype: int64

Unique timing labels: 4
Unique thresholds: 10
Target value counts:


realised_exceeds_threshold
1    36
0     4
Name: count, dtype: int64

Market-implied threshold probability range:


,market_implied_threshold_prob_raw,market_implied_threshold_prob_normalised,average_price_time_gap_minutes
count,40.000000,40.000000,40.000000
mean,0.830375,0.808516,2.656061
std,0.318659,0.307762,4.325846
min,0.085000,0.080416,0.100000
25%,0.733750,0.723481,0.137500
50%,0.977250,0.990442,0.235606
75%,1.040875,0.998199,2.754167
max,1.056500,0.999527,10.053030


,timing_label,forecast_timing_category,threshold_c,official_realised_temperature_c,realised_exceeds_threshold,market_implied_threshold_prob_normalised,lead_time_to_valid_hours,average_price_time_gap_minutes,forecast_feature_status
0,two_day_valid_time_example,full_day_ex_ante,24.0,32.6,1,0.998964,48.0,0.100000,pending_ai_weather_feature_merge
1,two_day_valid_time_example,full_day_ex_ante,25.0,32.6,1,0.996375,48.0,0.100000,pending_ai_weather_feature_merge
2,two_day_valid_time_example,full_day_ex_ante,26.0,32.6,1,0.993786,48.0,0.100000,pending_ai_weather_feature_merge
3,two_day_valid_time_example,full_day_ex_ante,27.0,32.6,1,0.988089,48.0,0.100000,pending_ai_weather_feature_merge
4,two_day_valid_time_example,full_day_ex_ante,28.0,32.6,1,0.984464,48.0,0.100000,pending_ai_weather_feature_merge
5,two_day_valid_time_example,full_day_ex_ante,29.0,32.6,1,0.948213,48.0,0.100000,pending_ai_weather_feature_merge
6,two_day_valid_time_example,full_day_ex_ante,30.0,32.6,1,0.901088,48.0,0.100000,pending_ai_weather_feature_merge
7,two_day_valid_time_example,full_day_ex_ante,31.0,32.6,1,0.730192,48.0,0.100000,pending_ai_weather_feature_merge
8,two_day_valid_time_example,full_day_ex_ante,32.0,32.6,1,0.403936,48.0,0.100000,pending_ai_weather_feature_merge
9,two_day_valid_time_example,full_day_ex_ante,33.0,32.6,0,0.191611,48.0,0.100000,pending_ai_weather_feature_merge


{'has_official_realised_temperature': True,
 'has_polymarket_event': True,
 'has_yes_markets': True,
 'has_price_history': True,
 'has_bin_price_table': True,
 'has_threshold_dataset': True}

## 11. Save outputs


In [23]:
bin_price_output_path = PROCESSED_DIR / "hong_kong_bin_prices_at_forecast_times.csv"
threshold_dataset_output_path = PROCESSED_DIR / "hong_kong_threshold_classification_dataset.csv"

bin_price_df.to_csv(bin_price_output_path, index=False)
threshold_dataset_df.to_csv(threshold_dataset_output_path, index=False)

print("Saved bin price table:", bin_price_output_path)
print("Saved threshold classification dataset:", threshold_dataset_output_path)


Saved bin price table: ../data/processed/threshold_dataset/hong_kong_bin_prices_at_forecast_times.csv
Saved threshold classification dataset: ../data/processed/threshold_dataset/hong_kong_threshold_classification_dataset.csv


## 12. Interpretation

This notebook constructs the first threshold-classification dataset for the project.

Each row represents a forecast information time and a temperature threshold. The realised target is based on the official Hong Kong Observatory settlement value, while the market-implied threshold probability is derived from Polymarket bin prices sampled at the forecast issue time or the first available price after that time.

The current dataset is not yet the final training dataset because AI weather forecast features still need to be merged. The forecast-temperature columns are therefore included as placeholders and marked as pending. The next empirical step is to attach AI or weather-model features at the same forecast issue times and then train a baseline probability model for threshold exceedance.

The output is nevertheless useful because it fixes the supervised-learning target, connects market prices to threshold probabilities and preserves the look-ahead-safe timing structure required for later scoring and trading tests.
